# Модуль-ноутбук: `evaluate`

Метрики, вибір порогу (з floor на support), калібрувальні графіки, bootstrap CI та демо label A-vs-C.

**Залежності:** `%run` 00_config.ipynb, 02_labels.ipynb

In [ ]:
%run 00_config.ipynb
%run 02_labels.ipynb

In [ ]:
"""Metrics, threshold selection, calibration, plots, and the label-A-vs-C fame demo.

Importable helpers used by train.py; also runnable (`notebooks/04_evaluate.ipynb`) to
re-render the comparison + plots from a freshly trained run.
"""

import json

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [ ]:
def _safe_auc(y, p):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else None

In [ ]:
def binary_metrics(y_true, p, t_high: float, t_low: float | None = None) -> dict:
    """Ranking + thresholded metrics. 'Post' decision = p >= t_high."""
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p, dtype=float)
    pred = (p >= t_high).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1]).tolist()
    m = {
        "roc_auc": _safe_auc(y_true, p),
        "pr_auc": float(average_precision_score(y_true, p)) if len(np.unique(y_true)) > 1 else None,
        "brier": float(brier_score_loss(y_true, p)) if len(np.unique(y_true)) > 1 else None,
        "precision_post": float(precision_score(y_true, pred, zero_division=0)),
        "recall_post": float(recall_score(y_true, pred, zero_division=0)),
        "f1_post": float(f1_score(y_true, pred, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "confusion_matrix": cm,  # [[TN, FP], [FN, TP]]
        "pos_rate": float(y_true.mean()),
        "n": int(len(y_true)),
        "t_high": float(t_high),
        "t_low": float(t_low) if t_low is not None else None,
    }
    if t_low is not None:  # abstention coverage
        post = p >= t_high
        dont = p <= t_low
        m["frac_post"] = float(post.mean())
        m["frac_dont"] = float(dont.mean())
        m["frac_unsure"] = float((~post & ~dont).mean())
    return m

In [ ]:
def bootstrap_ci(y_true, p, fn, n: int = 1000, seed: int = config.RANDOM_SEED):
    """Percentile bootstrap CI for a ranking metric fn(y, p)."""
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p, dtype=float)
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y_true))
    vals = []
    for _ in range(n):
        s = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y_true[s])) < 2:
            continue
        vals.append(fn(y_true[s], p[s]))
    if not vals:
        return (None, None)
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))

In [ ]:
def choose_thresholds(y_oof, p_oof, precision_target: float = config.POST_PRECISION_TARGET):
    """Pick (t_low, t_high) on TRAIN out-of-fold probs.

    t_high: smallest threshold whose 'Post' precision >= target (max recall among those).
    t_low : largest threshold whose 'Do not post' precision >= target.
    If the precision target is unreachable (weak signal), fall back to a PERCENTILE band
    on the OOF probabilities (commit only in the tails, abstain across the broad middle —
    the honest behaviour when the model can't be confident).
    Returns ((t_low, t_high), reached: bool).
    """
    y = np.asarray(y_oof).astype(int)
    p = np.asarray(p_oof, dtype=float)
    min_rec = config.THRESHOLD_MIN_RECALL
    min_sup = config.THRESHOLD_MIN_SUPPORT

    def _threshold_for(y_pos, score, target):
        """Smallest threshold whose precision >= target AND that has enough support
        (recall + predicted-positive count floors), so chance tail-points cannot qualify."""
        prec, rec, thr = precision_recall_curve(y_pos, score)
        prec, rec = prec[:-1], rec[:-1]  # align with thr
        n_pos = max(int(y_pos.sum()), 1)
        support = rec * n_pos / np.where(prec > 0, prec, np.nan)  # = predicted positives (TP/prec)
        ok = (prec >= target) & (rec >= min_rec) & (support >= min_sup)
        if not ok.any():
            return None
        cand = np.where(ok)[0]
        best = cand[np.argmax(rec[cand])]  # meet precision+support, maximise recall
        return float(thr[best])

    t_high = _threshold_for(y, p, precision_target)
    thr_neg = _threshold_for(1 - y, 1 - p, precision_target)
    t_low = None if thr_neg is None else float(1.0 - thr_neg)

    if t_high is None or t_low is None or not (0 < t_low < t_high < 1):
        # weak-signal fallback: confident only in the tails (~35/65 pct), Unsure in the middle
        t_low = float(np.quantile(p, 0.35))
        t_high = float(np.quantile(p, 0.65))
        if not (0 < t_low < t_high < 1):
            t_low, t_high = config.FALLBACK_BAND

    # Enforce a minimum confidence margin so "Unsure" stays meaningful on weak signal.
    m = config.MIN_DECISION_MARGIN
    t_high = float(max(t_high, 0.5 + m))
    t_low = float(min(t_low, 0.5 - m))

    # Honest `reached`: does the precision target ACTUALLY hold (with support) at the
    # FINAL clamped operating point we ship — not at some pre-clamp chance threshold?
    post = p >= t_high
    dont = p <= t_low
    reached = bool(
        post.sum() >= min_sup and y[post].mean() >= precision_target
        and dont.sum() >= min_sup and (1 - y[dont]).mean() >= precision_target
    ) if post.any() and dont.any() else False
    return (t_low, t_high), reached


# ----------------------------------------------------------------------------- plots

In [ ]:
def _plt():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    return plt

In [ ]:
def plot_calibration(curves: dict, path):
    """curves: name -> (y_true, p). Reliability diagram + per-model Brier."""
    from sklearn.calibration import calibration_curve
    plt = _plt()
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.plot([0, 1], [0, 1], "--", color="gray", label="perfect")
    for name, (y, p) in curves.items():
        if len(np.unique(y)) < 2:
            continue
        frac_pos, mean_pred = calibration_curve(y, p, n_bins=8, strategy="quantile")
        b = brier_score_loss(y, p)
        ax.plot(mean_pred, frac_pos, "o-", label=f"{name} (Brier={b:.3f})")
    ax.set_xlabel("mean predicted probability")
    ax.set_ylabel("observed fraction positive")
    ax.set_title("Calibration (temporal test set)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)

In [ ]:
def plot_roc_pr(curves: dict, path):
    from sklearn.metrics import precision_recall_curve as prc, roc_curve
    plt = _plt()
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.2))
    for name, (y, p) in curves.items():
        if len(np.unique(y)) < 2:
            continue
        fpr, tpr, _ = roc_curve(y, p)
        a1.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y,p):.2f})")
        pr, rc, _ = prc(y, p)
        a2.plot(rc, pr, label=f"{name} (AP={average_precision_score(y,p):.2f})")
    a1.plot([0, 1], [0, 1], "--", color="gray")
    a1.set(xlabel="FPR", ylabel="TPR", title="ROC")
    a2.axhline(list(curves.values())[0][0].mean(), ls="--", color="gray", label="base rate")
    a2.set(xlabel="recall", ylabel="precision", title="Precision-Recall")
    a1.legend(fontsize=8); a2.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)

In [ ]:
def plot_confusion(y, p, t_high, path):
    plt = _plt()
    cm = confusion_matrix(y, (np.asarray(p) >= t_high).astype(int), labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 3.6))
    ax.imshow(cm, cmap="Reds")
    ax.set_xticks([0, 1], ["pred: no", "pred: post"])
    ax.set_yticks([0, 1], ["true: no", "true: yes"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14)
    ax.set_title(f"Confusion @ t_high={t_high:.2f}")
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


# ----------------------------------------------------------------------------- comparison table

In [ ]:
def comparison_markdown(results: dict) -> str:
    cols = ["roc_auc", "pr_auc", "brier", "precision_post", "recall_post", "f1_post", "accuracy"]
    head = "| model | " + " | ".join(cols) + " |"
    sep = "| --- | " + " | ".join("---" for _ in cols) + " |"
    rows = [head, sep]
    for name, m in results.items():
        cells = []
        for c in cols:
            v = m.get(c)
            cells.append("n/a" if v is None else f"{v:.3f}")
        rows.append(f"| {name} | " + " | ".join(cells) + " |")
    return "\n".join(rows)


# ----------------------------------------------------------------------------- A vs C demo

In [ ]:
def label_a_vs_c_demo(raw_clean: pd.DataFrame, seed: int = config.RANDOM_SEED) -> dict:
    """Show WHY label C: a creator-only predictor has clearly MORE signal under label A
    (absolute top-quartile views, which tracks fame) than under label C (within-creator,
    where fame is ≈random). With only 4 creators the creator-mean predictor is
    granularity-limited (≤4 distinct scores), so the contrast is directional, not extreme."""
    from sklearn.model_selection import train_test_split

    df = raw_clean.dropna(subset=["play_count", config.CREATOR_COL]).reset_index(drop=True)
    tr, te = train_test_split(df, test_size=0.25, random_state=seed, shuffle=True)

    # label A: top-quartile pooled play_count (threshold from TRAIN)
    thr_a = tr["play_count"].quantile(0.75)
    ya_tr = (tr["play_count"] > thr_a).astype(int)
    ya_te = (te["play_count"] > thr_a).astype(int)
    rate_a = ya_tr.groupby(tr[config.CREATOR_COL].values).mean()
    pa = te[config.CREATOR_COL].map(rate_a).fillna(ya_tr.mean()).to_numpy()

    # label C: within-creator ER (threshold from TRAIN)
    thr_c = labels.fit_creator_thresholds(tr)
    yc_tr = labels.make_labels(tr, thr_c)
    yc_te = labels.make_labels(te, thr_c)
    okc = yc_te.notna().to_numpy()
    rate_c = yc_tr.groupby(tr[config.CREATOR_COL].values).mean()
    pc = te[config.CREATOR_COL].map(rate_c).fillna(yc_tr.mean()).to_numpy()

    auc_a = _safe_auc(ya_te.to_numpy(), pa)
    auc_c = _safe_auc(yc_te.to_numpy()[okc], pc[okc])
    return {
        "creator_only_auc_label_A": auc_a,
        "creator_only_auc_label_C": auc_c,
        "note": f"creator-only AUC ≈ {auc_a:.2f} under label A (fame partly predicts "
                f"top-quartile views) vs ≈ {auc_c:.2f} under label C (fame says ~nothing "
                f"about beating your own median). Directional support for C; the 4-creator "
                f"granularity caps how separable label A can look here.",
        "caveat": "random shuffle split + creator-base-rate predictor; illustrative only.",
    }

In [ ]:
def main() -> None:
    if config.METRICS_PATH.exists():
        print(config.METRICS_PATH.read_text())
    else:
        print("No metrics yet — run `notebooks/05_train.ipynb` first.")


if __name__ == "__main__":
    main()

In [ ]:
from types import SimpleNamespace
evaluate = SimpleNamespace(
    binary_metrics=binary_metrics,
    bootstrap_ci=bootstrap_ci,
    choose_thresholds=choose_thresholds,
    plot_calibration=plot_calibration,
    plot_roc_pr=plot_roc_pr,
    plot_confusion=plot_confusion,
    comparison_markdown=comparison_markdown,
    label_a_vs_c_demo=label_a_vs_c_demo,
    _safe_auc=_safe_auc,
    roc_auc_score=roc_auc_score,
    average_precision_score=average_precision_score,
)